<a href="https://colab.research.google.com/github/ddezouza/Data201_DanielaMelo/blob/main/Assignment2_DanielaMelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Part A - Core Wrangling (Method Chaining Required)



Using **one chained expression**, create a summary table that:

1. Keeps only observations where price > 250000

2. Keeps only homes with size > 1000

3. Creates a new variable: price_per_sqft = price / size

4. Groups by neighborhood

5. Computes:

  - mean of price_per_sqft
  - median of price_per_sqft
  - count of homes

6. Sorts the result by mean price_per_sqft(descending)

**Requirements**

- Do NOT create intermediate variables (df2, df3, etc.)

- Use .assign() to create new variables
- Use .agg() with named aggregation
- Do not use reset_index() (we did not cover it yet)

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/Reben80/Data201/refs/heads/main/Dataset/housing.csv")
df.head()

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily


In [7]:
(df #calling dataframe
  .query("price>250000 and size>1000") #filter by price and size using query
  .assign(price_per_sqft=lambda d: d["price"] / d["size"]) #use lambda to create price_per_sqft
  .groupby("neighborhood") #group by neighborhood
  .agg(mean_price_per_sqft=("price_per_sqft", "mean"), #compute mean price per sqft
       median_price_per_sqft=("price_per_sqft", "median"), #compute median price per sqft
       home_count=("price", "count")) #count homes
  .sort_values("mean_price_per_sqft", ascending=False) #sort by mean price per sqft descending
)

,mean_price_per_sqft,median_price_per_sqft,home_count
neighborhood,,,
Downtown,977.820905,1001.557049,99
Midtown,921.141446,901.992377,92
Suburb,861.917078,836.878644,157
Uptown,860.935608,843.653468,99
Waterfront,849.508891,792.663291,48


Apparently there's an alternate way to filter using lambda and pandas loc indexer (df.loc). I tried it and it seemed to work. also removed neighborhood as an index using as_index = False.

It worked really well. The table I used query for looks the same as the one I used .loc and lambda for. I think query is a lot easier to use though, but it's cool to see that we can use lambda for more than the assign option we covered.

In [9]:

(df #calling dataframe
  .loc[lambda d: (d["price"] > 250000) & (d["size"] > 1000)] #testing .loc and lambda
  .assign(price_per_sqft=lambda d: d["price"] / d["size"])
  .groupby("neighborhood", as_index=False) #remove neighborhood from index
  .agg(mean_price_per_sqft=("price_per_sqft", "mean"),
       median_price_per_sqft=("price_per_sqft", "median"),
       home_count=("price", "count"))
  .sort_values("mean_price_per_sqft", ascending=False)
)


,neighborhood,mean_price_per_sqft,median_price_per_sqft,home_count
0,Downtown,977.820905,1001.557049,99
1,Midtown,921.141446,901.992377,92
2,Suburb,861.917078,836.878644,157
3,Uptown,860.935608,843.653468,99
4,Waterfront,849.508891,792.663291,48


### Part B - Translation to dplyr

In a Markdown cell, write the equivalent dplyr code that performs the same transformation.

Your R pipeline should include:

- filter()
- mutate()
- group_by()
- summarise()
- arrange()


I included the R code and libraries that I felt gave me the most similar result.
```R
library(dplyr)
library(readr)
library(tibble)

df <- read_csv("housing.csv")

df %>%
  filter(price > 250000, size > 1000) %>%
  mutate(price_per_sqft = price / size) %>%
  group_by(neighborhood) %>%
  summarise(
    mean_price_per_sqft   = mean(price_per_sqft, na.rm = TRUE),
    median_price_per_sqft = median(price_per_sqft, na.rm = TRUE),
    home_count = n()) %>%
  arrange(desc(mean_price_per_sqft)) %>%
  as_tibble()

```

**Reflection**

I feel both pandas and dplyr are pretty clear in terms of syntax. You can more or less tell the purpose of each line of code in each case by looking at the elements in each line so I don't think either feels clearer than the other for me. Both have specific functions to achieve specific purposes and I feel they follow a similar order in terms of what is natural to do first when creating a summary table. I do feel like it is a bit easier to type out the python code because the R pipe (%>%) is a little less natural than just typing out the indent .formula format that pandas uses, but I also feel I'm a little more aware of the steps I'm chaining in R because the pipe is really easy to spot and recognize. Overall I find the two methods pretty equivalent and don't have a particular preference between the two.

### Part C - Boolean Logic Debugging

In [10]:
df[df["price"] > 250000 & df["size"] > 1000]

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

1. Fix the code

In [11]:
df[(df["price"] > 250000) & (df["size"] > 1000)]

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,1500000,1443.241197,3.0,Midtown,Condo
596,100597,1500000,1083.909714,2.0,Suburb,Condo
597,100598,1500000,1600.126432,1.0,Suburb,SingleFamily
598,100599,1500000,1248.216637,1.0,Waterfront,Condo


2. Explain why the error occurs.

Without the parentheses pandas cannot interpret the two parts of the filter we're trying to apply. We want to filter for houses that both cost more than 250000 and are larger than 1000 square feet, but pandas cannot understand what makes a complete expression if we don't explicitly outline it. It tries to read both parts of the argument as a single expression so we have to outline the two parts of the filter we want with parentheses so each condition is read as a complete expression and pandas understands we want to apply both of them at the same time (&).

3. Rewrite the filter using .query() instead.

In [12]:
df.query("price>250000 and size>1000")

,listing_id,price,size,bedrooms,neighborhood,type
0,100001,1500000,1280.741760,1.0,Suburb,Townhouse
1,100002,1500000,1406.283113,2.0,Uptown,SingleFamily
2,100003,1500000,4146.825713,6.0,Suburb,MultiFamily
3,100004,1500000,3946.599818,6.0,Suburb,SingleFamily
4,100005,1500000,1243.751760,1.0,Downtown,MultiFamily
...,...,...,...,...,...,...
595,100596,1500000,1443.241197,3.0,Midtown,Condo
596,100597,1500000,1083.909714,2.0,Suburb,Condo
597,100598,1500000,1600.126432,1.0,Suburb,SingleFamily
598,100599,1500000,1248.216637,1.0,Waterfront,Condo


### Part D - Short Concept Questions

1. Why must we wrap each condition in parentheses using & in pandas?

Because otherwise pandas is unable to interpret both conditions as separate and we get an error. Each condition needs to be considered a complete expression and the & denotes we want both conditions applied.

2. What is the advantage of method chaining over creating many temporary dataframes?

It's quicker and more readable. You don't have to keep track of multiple data frames.

3. In ```.agg(mean_price=("price","mean"))```, what does ```"price"``` represent? What does ```"mean"``` represent?

```"price"``` represents the variable we're calculating with, and ```"mean"``` represents what calculation we're performing. We're computing the mean of the variable 'price'.

4. When you ```groupby("neighborhood")```, why does ```neighborhood``` appear on the left (index) in the result table?

Because the variable 'neighborhood' is being used as the index through which the other varibles are being grouped. Pandas understands that you want to use the grouping variable as the new index for you table and makes it easier to select groups based on that variable by making it an index.



### Optional Extension

Create a second summary table grouped by ```type``` that reports:

- mean price
- median price
- count of listings

In [14]:
(df #calling dataframe
  .groupby("type") #group by type
  .agg(mean_price=("price", "mean"), #compute mean price
       median_price=("price", "median"), #compute median price
       home_count=("price", "count")) #count homes
)

,mean_price,median_price,home_count
type,,,
Condo,1500000.0,1500000.0,183
MultiFamily,1500000.0,1500000.0,63
SingleFamily,1500000.0,1500000.0,235
Townhouse,1500000.0,1500000.0,119
